<a href="https://colab.research.google.com/github/Kamindumenula/SE4050-Household-Waste-Classification-System/blob/main/resnet50/IT23293908resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


#DOWNLOAD & EXTRACT DATASET

if not os.path.exists("SE4050-Household-Waste-Classification-System"):
    !git clone https://github.com/Kamindumenula/SE4050-Household-Waste-Classification-System.git

DATASET_DIR = "/content/dataset-resized"
if not os.path.exists(DATASET_DIR):
    !unzip -q -o SE4050-Household-Waste-Classification-System/dataset/archive.zip -d /content/


# LOAD & SPLIT (80% TRAIN, 10% VAL, 10% TEST)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

# Load 80% for training
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Load 20% held-out
val_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Split the 20% into 10% val and 10% test
val_size = int(len(val_test_ds) * 0.5)
val_ds = val_test_ds.take(val_size)
test_ds = val_test_ds.skip(val_size)

# APPLY COMMON DATA AUGMENTATION (ONLY TO TRAIN DATASET!)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),  # Flips left/right
    layers.RandomRotation(0.15),      # Rotates ±15%
    layers.RandomZoom(0.15),          # Zooms in/out ±15%
    layers.RandomContrast(0.1),       # Slight lighting change
], name="common_data_augmentation")

# We apply augmentation directly to train_ds:
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)


# SPEED OPTIMIZATION (CACHE & PREFETCH)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

classes = sorted(os.listdir(DATASET_DIR))
print(f"Setup Complete!")
print(f"Classes ({len(classes)}): {classes}")
print(f"Batches -> Train: {len(train_ds)} (Augmented) | Val: {len(val_ds)} | Test: {len(test_ds)}")

Cloning into 'SE4050-Household-Waste-Classification-System'...
remote: Enumerating objects: 203, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 203 (delta 59), reused 68 (delta 25), pack-reused 75 (from 1)
Receiving objects: 100% (203/203), 66.39 MiB | 28.43 MiB/s, done.
Resolving deltas: 100% (71/71), done.
Found 2527 files belonging to 6 classes.
Using 2022 files for training.
Found 2527 files belonging to 6 classes.
Using 505 files for validation.
Setup Complete!
Classes (6): ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Batches -> Train: 64 (Augmented) | Val: 8 | Test: 8


In [2]:
from tensorflow.keras.applications import ResNet50

# Load ResNet50 model without the top layers
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base layers
base_model.trainable = False

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
from tensorflow.keras import layers
from tensorflow import keras

NUM_CLASSES = 6

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,758 (90.98 MB)

 Trainable params: 263,046 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [4]:
# 1. Compile the Model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 2. Train the Model
epochs = 10
print("Training started...")
history = model.fit(
    train_ds,                  # Your training data variable
    validation_data=val_ds,    # Your validation data variable
    epochs=epochs
)

# 3. Save the Model in Colab
model.save('IT23293908ResNet50.keras')
print("Model saved successfully!")

Training started...
Epoch 1/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 388s 6s/step - accuracy: 0.6518 - loss: 0.9451 - val_accuracy: 0.7734 - val_loss: 0.6397
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 379s 6s/step - accuracy: 0.7923 - loss: 0.5510 - val_accuracy: 0.7773 - val_loss: 0.6115
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 375s 6s/step - accuracy: 0.8393 - loss: 0.4475 - val_accuracy: 0.7930 - val_loss: 0.5995
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 381s 6s/step - accuracy: 0.8546 - loss: 0.4004 - val_accuracy: 0.8164 - val_loss: 0.4905
Epoch 5/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 382s 6s/step - accuracy: 0.8947 - loss: 0.3065 - val_accuracy: 0.8242 - val_loss: 0.4875
Epoch 6/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 385s 6s/step - accuracy: 0.8981 - loss: 0.2715 - val_accuracy: 0.8320 - val_loss: 0.4763
Epoch 7/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 381s 6s/step - accuracy: 0.9041 - loss: 0.2765 - val_accuracy: 0.8242 - val_loss: 0.5174
Epoch 8/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 374s 6s/step - accuracy: 0.9189 - loss: 0.2406 - val_